# 13. ETAS I：條件強度、分支過程與模擬

{doc}`第 12 章 <12_clustering_laws>`把叢集的三條老經驗律推到了它們該有
的精確度，最後留下三個零件：$\kappa(m)$ 管「一個事件生幾個後代」、
{eq}`eq:omori-density` 的 $g(t)$ 管「後代什麼時候出生」、$f(x,y;m)$
管「後代出生在哪裡」。{doc}`第 10 章 <10_point_process>`則給了寫下模型
的語法：只要寫得出條件強度 {eq}`eq:cond-int`，概似 {eq}`eq:pp-loglik`、
殘差 {eq}`eq:time-rescale`、模擬與檢驗就會自動跟著出現。

這一章要做的事只有一步：**把三個零件塞進條件強度的求和號裡**。做完這
一步，得到的東西叫 ETAS（Epidemic-Type Aftershock Sequence，傳染型餘震
序列模型，Ogata 1988），是目前世界各國作業化地震預報的主力引擎，也是
所有新模型都必須先贏過的基準線。

但這一步遠不只是「把三條律寫在一起」。疊加會產生三件模型設計者沒有
明文寫進去的東西：**二次餘震**、**前震**，以及一個叫**分支比**的數字
——它同時決定序列會不會熄滅、目錄裡有多少比例是餘震，以及整個模型的
概似積分收不收斂。這一章的重點就是把這三件湧現的東西推導清楚。

分工上，本章只管**模型結構與模擬**：條件強度長什麼樣、參數各自負責
什麼、怎麼從模型生出一份目錄。**參數怎麼從真實目錄估出來**（概似的
數值實作、隨機除叢的 $\rho_{ij}$ 與 $\phi_j$、simplETAS 釘死參數的
理由、R–J 模型是哪一種特例）全部留給下一章。會用到的舊式子只有四條：
{eq}`eq:cond-int`、{eq}`eq:pp-loglik`、{eq}`eq:omori-density`，以及第
11 章的 $\beta = b\ln 10$。

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

## 13.1 疊加這一步

修正 Omori 律描述的是**一個**主震之後的餘震率。它預設了一件事：序列裡
有一個特別的事件（主震），其他事件都是它的附屬品，而且附屬品之間沒有
互動。這個預設在真實資料上撐不了多久。1999 集集、2024 花蓮 0403 的
序列都看得到同一個現象：主震衰減的途中冒出一個 $M_L\,6$ 的大餘震，
接下來幾天的活動又跳回高點、再重新衰減一次。

傳統作法是手動判斷：這是主震的餘震，還是餘震的餘震？要不要另外擬一條
Omori 曲線？曲線的起算時刻放哪裡？每一個問題都是一個主觀選擇，而第 12
章已經示範過，主觀選擇會在最後的參數上留下指紋。

ETAS 的解法乾脆得多，乾脆到有點粗暴：**不要判斷。讓每一個事件都掛上
自己的觸發核，全部疊加起來。**

寫成條件強度就是（先只看時間，空間留到 13.2）：

$$\lambda^*(t) \;=\; \mu \;+\; \sum_{i:\,t_i < t}
  \kappa(m_i)\;g(t - t_i)$$

求和號裡的兩個因子分工明確：$\kappa$ 管「生幾個」，$g$ 管「什麼時候」。
三項逐一讀：

- $\mu$ 是**背景率**（次／天）。它代表沒有被目錄裡任何事件觸發、由板塊
  加載慢慢累積出來的「自發」地震。它是常數，不隨歷史變動——這是 ETAS
  最強、也最常被質疑的一條假設（13.10 節會回來談）。
- $\kappa(m_i) = A\,e^{\alpha(m_i-m_0)}$ 是事件 $i$ 的**產能**，也就是
  它平均會產生幾個**直接**後代（12.4 節）。
- $g(t-t_i)$ 是{eq}`eq:omori-density`的正規化 Omori 密度，決定這些後代
  的出生時刻怎麼分布。因為它積分為 1，「量」全部收在 $\kappa$ 裡、
  「形狀」全部收在 $g$ 裡，兩者互不污染。

### 求和號跑遍誰，決定了這個模型的一切

這條式子與修正 Omori 律的差別，只在求和號底下那一行小字：$i$ 跑遍
**$t$ 之前的全部事件**，不是「主震」。差別看起來微不足道，後果卻是
結構性的。

假設主震後第 3 天出現一個 $M_L\,6.0$ 的餘震。從第 3 天起，這個事件就
以自己的 $\kappa(6.0)\,g(t-3)$ 加進強度裡，跟主震享有完全一樣的地位。
它的後代出生之後，又各自加進強度裡。**沒有任何一行程式寫過「二次
餘震」這四個字，二次餘震卻必然出現**，而且出現的比例完全由 $\kappa$
與 $g$ 決定，不需要新參數。

疊加這一步同時取消了兩個概念：主震與餘震。ETAS 的世界裡只有兩種事件
——**背景事件**（來自 $\mu$ 那一項）與**被觸發事件**（來自 $\sum$ 那一
項）。「主震」不是模型的輸入，甚至不是模型的輸出，它只是我們事後在
一堆事件裡挑出最大的那一個時貼上去的標籤。第 12 章證明過這條標籤會
污染 b 值；ETAS 的回答是根本不貼。

用第 10 章的分類法看，這是一個 **self-exciting**（自我激發）點過程，
也就是 Hawkes（1971）過程的一支：觸發核帶了規模標記、規模又服從 GR
律。第 10 章說「三種記憶結構」時舉的例子就是它。

先直接看一份模擬出來的目錄長什麼樣。下面用 13.6 節要正式介紹的分支法
生成一份 180 天的時間型 ETAS 目錄，上圖畫條件強度、下圖畫事件本身：

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from gdms_toolkit.viz import ACCENT, PALETTE, QUAKE_COLOR, apply_layout

M0, M_MAX = 3.0, 8.0                 # 模型門檻規模、規模上限
BETA = np.log(10.0)                  # GR 斜率（b = 1）
C_OM, P_OM = 0.01, 1.3               # Omori 核參數（天）
DM = M_MAX - M0                      # 截斷寬度


def A_from_n(n_ratio, alpha):
    """由分支比反推產能尺度 A（截斷 GR 版本，推導見 13.3）。"""
    d = BETA - alpha
    inner = (1 - np.exp(-d * DM)) / d if abs(d) > 1e-9 else DM
    return n_ratio * (1 - np.exp(-BETA * DM)) / (BETA * inner)


def draw_mag(rng, size):
    """截斷 GR 抽樣（反函數法，見 10.6）：m 落在 [M0, M_MAX]。"""
    u = rng.random(size)
    return M0 - np.log1p(-u * (1 - np.exp(-BETA * DM))) / BETA


def simulate_etas(mu, n_ratio, alpha, T, seed):
    """時間型 ETAS 的分支法模擬（虛擬碼見 13.6）。

    回傳 (目錄 DataFrame, 產能尺度 A)。DataFrame 欄位：
    t 時間、m 規模、parent 親代索引（背景事件為 −1）、
    gen 世代（背景為 0）、root 所屬家族的始祖索引。
    """
    rng = np.random.default_rng(seed)
    A = A_from_n(n_ratio, alpha)
    n_bg = rng.poisson(mu * T)                       # 第 0 代：背景事件
    t = list(rng.uniform(0, T, n_bg))
    m = list(draw_mag(rng, n_bg))
    parent, gen, root = [-1] * n_bg, [0] * n_bg, list(range(n_bg))
    todo = list(range(n_bg))
    while todo:                                      # 遞迴繁殖
        i = todo.pop()
        k = rng.poisson(A * np.exp(alpha * (m[i] - M0)))
        if k == 0:
            continue
        dt = C_OM * ((1 - rng.random(k)) ** (-1 / (P_OM - 1)) - 1)
        mk = draw_mag(rng, k)
        for d, mv in zip(dt, mk):
            if t[i] + d < T:                         # 落在窗外的直接丟棄
                t.append(t[i] + d)
                m.append(mv)
                parent.append(i)
                gen.append(gen[i] + 1)
                root.append(root[i])
                todo.append(len(t) - 1)
    cat = pd.DataFrame(dict(t=t, m=m, parent=parent, gen=gen, root=root))
    order = np.argsort(cat.t.to_numpy())          # 依時間排序並重編索引
    newpos = np.empty(len(cat), dtype=int)
    newpos[order] = np.arange(len(cat))
    cat = cat.iloc[order].reset_index(drop=True)
    par = cat.parent.to_numpy()
    cat["parent"] = np.where(par >= 0, newpos[np.maximum(par, 0)], -1)
    cat["root"] = newpos[cat.root.to_numpy()]
    return cat, A


def lam_star(tt, cat, mu, A, alpha):
    """時間型 ETAS 的條件強度曲線（逐事件疊加）。"""
    lam = np.full_like(tt, float(mu))
    for ti, mi in zip(cat.t.to_numpy(), cat.m.to_numpy()):
        msk = tt > ti
        lam[msk] += (A * np.exp(alpha * (mi - M0)) * (P_OM - 1) / C_OM
                     * (1 + (tt[msk] - ti) / C_OM) ** -P_OM)
    return lam


MU1, N1, ALPHA1, T1 = 1.0, 0.85, 1.2, 180.0
cat1, A1 = simulate_etas(MU1, N1, ALPHA1, T1, seed=10)
tt = np.linspace(0, T1, 3000)
lam1 = lam_star(tt, cat1, MU1, A1, ALPHA1)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.45, 0.55], vertical_spacing=0.04)
fig.add_trace(go.Scatter(x=tt, y=lam1, mode="lines", name="條件強度 λ*(t)",
                         line=dict(color=ACCENT, width=1.4)), row=1, col=1)
fig.add_trace(go.Scattergl(x=cat1.t, y=cat1.m, mode="markers", name="模擬事件",
                           marker=dict(size=4 + (cat1.m - M0) * 3.5,
                                       color=QUAKE_COLOR, opacity=0.6)),
              row=2, col=1)
fig.update_yaxes(title_text="λ*（次/天）", type="log", row=1, col=1)
fig.update_yaxes(title_text="規模", row=2, col=1)
fig.update_xaxes(title_text="時間（天）", row=2, col=1)
apply_layout(fig, height=540, hovermode="x",
             title=f"模擬的時間型 ETAS 目錄（n = {N1}，α = {ALPHA1}，"
                   f"共 {len(cat1)} 個事件，其中背景 "
                   f"{int((cat1.gen == 0).sum())} 個）")
fig

上圖是對數軸的條件強度。每次較大事件出現，$\lambda^*$ 立刻跳升一大截、
再按 Omori 律衰減；衰減途中若又冒出一個大事件，強度再度跳升。整份目錄
裡沒有任何一行程式碼寫過「序列」，但主震–餘震的樣子自己長了出來，而且
長出了好幾個層次：大叢集裡面套著小叢集。

圖說裡的兩個數字值得記住：全部事件數與背景事件數。兩者的比值不是巧合
——13.4 節會證明它由分支比決定，理論上是 $1/(1-n)$，並說明為什麼有限
觀測窗會讓實測值系統性偏低。

## 13.2 三個零件與正規化

13.1 節的式子還缺一半：空間，以及規模。這一節把完整的時空 ETAS 拼出來，
順便處理一個所有人都會踩的坑——**參數的意義綁在正規化慣例上，換一個
慣例就換一個數字**。

### 為什麼一定要正規化

觸發核要回答三個獨立的問題：生幾個、什麼時候、在哪裡。三個答案應該由
三組互不干擾的參數負責。但 Ogata 原始寫法裡的時間核

$$n(t) = K\,(t + c)^{-p}$$

不是機率密度——它的積分是 $K c^{1-p}/(p-1)$（12.1 節推過），依賴 $c$ 與
$p$。這代表「生幾個」這件事被 $K$、$c$、$p$ 三個參數共同決定：你調
$p$，總後代數也跟著變。這在數學上完全合法，在解讀上是災難。

本書（以及 Jalilian 2019、simplETAS 等現代寫法）一律採用正規化版本：
時間核用{eq}`eq:omori-density`的密度 $g$，空間核 $f$ 也正規化為密度，
所有的「量」全部收進 $\kappa$。這樣一來 $\kappa$ 就是乾乾淨淨的**期望
直接後代數**，是一個無量綱的數字，可以直接跟「一個病例平均感染幾人」
相提並論。

### $K$ 與 $A$ 的換算，以及為什麼不能直接比較

把兩種寫法擺在一起。Ogata 式：

$$\lambda^*(t) = \mu + \sum_i K\,e^{\alpha(m_i-m_0)}\,(t-t_i+c)^{-p}$$

正規化式（本書）：

$$\begin{aligned}
\lambda^*(t) &= \mu + \sum_i A\,e^{\alpha(m_i-m_0)}\,
  \frac{p-1}{c}\Bigl(1 + \frac{t-t_i}{c}\Bigr)^{-p} \\
  &= \mu + \sum_i A\,(p-1)\,c^{\,p-1}\,e^{\alpha(m_i-m_0)}\,
     (t-t_i+c)^{-p} .
\end{aligned}$$

第二行只是把 $\frac{p-1}{c}\bigl(1+\frac{t-t_i}{c}\bigr)^{-p}$ 的 $c$
提出來：$\bigl(\frac{t-t_i+c}{c}\bigr)^{-p} = c^{\,p}(t-t_i+c)^{-p}$，
再除以 $c$ 得 $c^{\,p-1}$。逐項比對係數，立刻得到換算式：

$$K = A\,(p-1)\,c^{\,p-1}
  \qquad\Longleftrightarrow\qquad
  \frac{A}{K} = \frac{c^{\,1-p}}{p-1} .$$

這條式子有兩個要命的性質。**第一，比值依賴 $p$ 與 $c$**，所以兩篇論文
的 $K$ 相同不代表產能相同，$A$ 相同也不代表 $K$ 相同。**第二，比值大得
離譜**，不是「差一點點」可以忽略的等級：

| $p$ | $c$（天） | $A/K$ | 出處脈絡 |
|---|---|---|---|
| 1.05 | 0.001 | 28.3 | 高密度網、$c$ 壓得很小 |
| 1.10 | 0.010 | 15.8 | 台灣 ETAS 估計的量級 |
| 1.15 | 0.005 | 14.8 | simplETAS 的釘死值 |
| 1.30 | 0.010 | 13.3 | 本章模擬所用 |

把別人的 $K$ 當成你的 $A$ 讀，產能會差十幾到三十倍——而分支比正比於
$A$，等於整個模型的臨界性判斷全錯。**跨論文比較產能之前，先確認三件
事：時間核有沒有正規化、$m_0$ 是多少、$c$ 與 $p$ 是多少。**

這還沒完。$A$ 本身也綁在 $m_0$ 上（$\kappa(m_0)=A$，「門檻事件的產能」），
所以連正規化寫法之間，$m_0$ 不同時 $A$ 也不能直接比。13.3 節末會把這件
事量化。

### 完整的時空 ETAS

現在把空間與規模補上。第 10 章 10.4 節的**可分離性**假設說，規模與
時空可以拆開：規模由 $s(m)=\beta e^{-\beta(m-m_0)}$ 獨立決定，與歷史
無關。於是四維的條件強度寫成

$$\lambda^*(t,x,y,m) \;=\; s(m)\left[\mu(x,y)
  + \sum_{i:\,t_i<t} \kappa(m_i)\,g(t-t_i)\,
    f(x-x_i,\,y-y_i;\,m_i)\right]$$ (eq:etas-intensity)

三個零件全部來自第 12 章：

$$\begin{aligned}
\kappa(m) &= A\,e^{\alpha(m-m_0)}, \\
g(t) &= \frac{p-1}{c}\Bigl(1+\frac{t}{c}\Bigr)^{-p}, \\
f(x,y;m) &= \frac{q-1}{\pi D e^{\gamma(m-m_0)}}
  \left[1 + \frac{r^2}{D e^{\gamma(m-m_0)}}\right]^{-q},
  \qquad r^2 = x^2+y^2 .
\end{aligned}$$

{eq}`eq:etas-intensity` 是本章的擁有物，第 14 章之後一律引用不重寫。
它有一個值得停一秒的結構：**方括號裡的東西完全不含 $m$**。這就是可
分離性的實際樣子——「接下來會有幾個地震」由歷史決定，「這個地震有多
大」不由歷史決定。$s(m)$ 只是乘在外面的一個因子，在概似裡會整個分離
出去（第 14 章會用到這件事）。

### 八個參數，各自負責什麼

扣掉可以單獨估的 $\beta$，時空 ETAS 有八個參數：

| 參數 | 出現在 | 管什麼 | 典型值 |
|---|---|---|---|
| $\mu$ | 背景項 | 自發地震的率（或空間場的尺度） | 依區域 |
| $A$ | $\kappa$ | 門檻事件的產能 | 0.01–0.3 |
| $\alpha$ | $\kappa$ | 規模換算成產能的效率 | 0.35–3.1 |
| $c,\,p$ | $g$ | 時間核的平緩期與衰減陡度 | 見 12.9 |
| $D,\,\gamma,\,q$ | $f$ | 空間尺度、尺度隨規模成長、遠場冪次 | 見 12.9 |

八個之中真正**只**屬於這一章的討論是 $\alpha$：它是 ETAS 少數能直接
對應「活動型態」的參數。Ogata（1992）在日本目錄上量到的分界很乾淨
——**群震（swarm）型活動 $\alpha\in[0.35,0.85]$，非群震活動
$\alpha\in[1.2,3.1]$**。$\alpha$ 小代表大小地震的觸發能力差不多，序列
看起來像一群規模相近的事件擠在一起；$\alpha$ 大代表大地震壓倒性主導，
序列看起來就是典型的主震–餘震。13.6 節會用模擬把這兩種樣貌並排。

### Ogata & Zhuang（2006）：把空間尺度自產能解耦

上面的八參數版本不是一開始就長這樣。Ogata（1998）的時空 ETAS 只有七個
參數，因為它假設

$$\kappa(m) \;\propto\; \sigma(m) \;\propto\; e^{\alpha(m-m_0)},$$

也就是**同一個 $\alpha$ 同時決定「生幾個」與「散多遠」**。這個綁定有它
的道理：兩者都隨規模指數成長，而且直覺上大地震既多產又影響範圍大。
問題是，指數成長的**速率**沒有理由相同。產能來自宇津的產能律，空間
尺度來自 Utsu–Seki 律（12.4 節），這是兩條獨立歸納出來的經驗律。

診斷證據來自 Zhuang et al.（2004）。他們用隨機除叢把每個事件的親代
機率算出來，然後畫一張診斷圖：橫軸親代規模、縱軸子代距離的眾數。若
綁定成立，這張圖的斜率應該等於 $\hat\alpha$。實測結果是**斜率明顯比
$\hat\alpha$ 平緩**——也就是說，大地震確實多產，但它的餘震區沒有大到
產能所暗示的程度。綁定造成系統性偏差。

Ogata & Zhuang（2006）因此把兩者拆開，引入獨立的 $\gamma$ 專管空間
尺度，參數由七個變八個。改用解耦版本之後，同一張診斷圖上的系統性
偏差幾乎消失。他們對 JMA 三個構造背景不同的資料集（1926–1995）做
AIC 比較，解耦版本一律勝出，估出來的量級是 $\hat p\approx1.03$–1.05、
$\hat q\approx1.58$–1.74、$\hat\alpha\approx1.1$–1.65、
$\hat\gamma\approx0.80$–1.33。

這件事的教訓比參數本身重要：**一個參數兼兩份差事，擬合可能還過得去，
但診斷圖會出賣它**。ETAS 的參數不是靠 AIC 一路加上去的，$\gamma$ 是
先看到偏差、再加參數修正——這是模型發展的正確順序。

順帶一提，Ogata（1998）的空間核其實還帶一個 $2\times2$ 正定對稱矩陣
$S_j$，用來描述餘震區的橢圓形狀（斷層走向、傾角、定位誤差都揉在裡面）。
本書為了教學清晰採用各向同性版本，但要記得**各向同性只是簡化**，
13.8 節的家族表會提到把它換掉的各種變體。

### 台灣的兩份獨立估計

台灣已經有自己的時空 ETAS 參數，而且有兩份**互相獨立**的估計。中央
氣象署 112 年委辦計畫用 1994–2021 目錄（$M_c=3.0$）得到 $p=1.06$、
$\alpha=1.17$；2025 年大埔地震的快報（Hsieh et al. 2025）用另一套
流程得到 $p=1.04$、$\alpha=1.04$。兩組數字高度一致，互為交叉驗證
——這就是「台灣的 ETAS 長什麼樣」的基準值，也是第 12 章 12.9 節那張
表裡「$p=1.04$–1.06」那一列的來源。

兩個 $\alpha$ 都落在 1.0–1.2，正好在 Ogata（1992）「非群震」區間的
下緣。這代表台灣整體偏主震–餘震型，但不極端。至於這些數字是怎麼估
出來的、標準誤有多大、早期目錄不完整會把 $\alpha$ 壓低多少，全部是
{doc}`第 14 章 <14_etas_estimation>`的主題。

## 13.3 分支比：一個積分決定模型的生死

現在來推導本章最重要的一個數字。

$\kappa(m)$ 給的是「**規模 $m$ 的**事件平均生幾個後代」，這依賴 $m$。
但我們常常要問一個不依賴 $m$ 的問題：**隨便挑一個事件，它平均生幾個
後代？** 答案要對規模取期望。而規模服從 $s(m)$，於是

$$n \;\equiv\; \int_{m_0}^{\infty} \kappa(m)\,s(m)\,\mathrm{d}m .$$

這個 $n$ 叫**分支比**（branching ratio）。代入兩個零件：

$$\begin{aligned}
n &= \int_{m_0}^{\infty} A\,e^{\alpha(m-m_0)}\cdot
     \beta\,e^{-\beta(m-m_0)}\,\mathrm{d}m \\
  &= A\beta \int_{0}^{\infty} e^{-(\beta-\alpha)u}\,\mathrm{d}u
     \qquad (u = m - m_0) \\
  &= A\beta \left[\frac{e^{-(\beta-\alpha)u}}{-(\beta-\alpha)}
     \right]_{0}^{\infty}
   = \frac{A\beta}{\beta-\alpha} .
\end{aligned}$$ (eq:branching-ratio)

收斂條件全壓在那個上限上：$u\to\infty$ 時 $e^{-(\beta-\alpha)u}\to0$
需要 $\beta-\alpha>0$。所以

$$n = \frac{A\beta}{\beta-\alpha}\,,\qquad \alpha < \beta .$$

**$\alpha<\beta$ 不是經驗觀察，是積分收斂的條件**——結構上與 12.1 節
的 $p>1$、$q>1$ 完全平行。三者都在說同一件事：ETAS 的每一個零件都
必須「積得出有限的量」，否則模型無法定義。

這個條件的物理意義也很清楚。$\alpha>\beta$ 代表產能隨規模成長得比
「大事件變罕見」的速度還快，於是期望後代數被稀有的巨大事件主宰，積分
發散。用第 12 章的語言說：**規模分布的尾巴太肥，產能律的尾巴更肥，
兩者相乘就爆掉了。**

### 有限 $M_{\max}$ 之下的分支比

上面的積分積到 $m=\infty$，但真實地球顯然有規模上限：一條斷層放不出
任意大的地震。把 GR 律截斷在 $M_{\max}$，規模密度變成

$$s_T(m) = \frac{\beta\,e^{-\beta(m-m_0)}}{1 - e^{-\beta\Delta M}},
  \qquad m_0 \le m \le M_{\max},\quad \Delta M \equiv M_{\max}-m_0 .$$

分母就是原密度在 $[m_0,M_{\max}]$ 上的積分（$\int_0^{\Delta M}\beta
e^{-\beta u}\mathrm{d}u = 1-e^{-\beta\Delta M}$），除掉它就重新歸一。
重做一次分支比的積分：

$$\begin{aligned}
n &= \int_{m_0}^{M_{\max}} A\,e^{\alpha(m-m_0)}\,s_T(m)\,\mathrm{d}m \\
  &= \frac{A\beta}{1-e^{-\beta\Delta M}}
     \int_{0}^{\Delta M} e^{-(\beta-\alpha)u}\,\mathrm{d}u \\
  &= \frac{A\beta}{1-e^{-\beta\Delta M}}\cdot
     \frac{1 - e^{-(\beta-\alpha)\Delta M}}{\beta-\alpha} .
\end{aligned}$$

兩個檢查。**其一**，令 $\Delta M\to\infty$ 且 $\alpha<\beta$，兩個
指數項都趨於 0，回到 {eq}`eq:branching-ratio`——截斷版本包含未截斷
版本作為極限，沒有矛盾。**其二**，這條式子對 $\alpha>\beta$ 也有定義：
此時 $(\beta-\alpha)<0$，分子分母同時變號，結果仍是正的有限值。
換句話說，**截斷解除了 $\alpha<\beta$ 的限制**——代價是 $n$ 從此依賴
一個你必須自己指定的 $M_{\max}$。

### $\alpha=\beta$：simplETAS 為什麼仍然收斂

最有意思的是 $\alpha=\beta$ 這個邊界。未截斷版本在這裡直接除以零；
但截斷版本的積分完全沒有問題，因為被積函數退化成常數 1：

$$\int_{0}^{\Delta M} e^{0}\,\mathrm{d}u = \Delta M
  \qquad\Longrightarrow\qquad
  n \;=\; \frac{A\beta\,\Delta M}{1 - e^{-\beta\Delta M}} .$$

這正是 simplETAS（Mancini & Marzocchi 2023）敢把 $\alpha$ 釘死在
$\beta=\ln10$ 的技術前提。$\alpha=\beta$ 的意思是「產能隨規模成長的
速率，恰好等於大事件變罕見的速率」，也就是**完全自相似**：每一個規模
級距對總觸發量的貢獻都一樣。這是很漂亮的設定，但它讓分支比失去了
「內生的」收斂機制，必須靠外部指定的 $M_{\max}$ 才有限。

量化一下這個依賴有多強。simplETAS 在義大利校正得到 $A=0.047$、
$m_0=3.95$，取 $\beta=\ln10$：

| $M_{\max}$ | $\Delta M$ | $n$ | 說明 |
|---|---|---|---|
| 7.5 | 3.55 | 0.38 | 保守的區域上限 |
| 8.0 | 4.05 | 0.44 | 常用的中間值 |
| 8.5 | 4.55 | 0.49 | 寬鬆的上限 |

（這三列的 $e^{-\beta\Delta M}$ 都小於 $10^{-3}$，分母幾乎等於 1，
$n\approx A\beta\Delta M$ 隨 $\Delta M$ 線性成長。）**$M_{\max}$ 改
一級，$n$ 就差三成**。這不是模型的缺陷，而是一個必須明講的建模選擇：
**報告 $\alpha=\beta$ 的 ETAS 分支比時，不寫 $M_{\max}$ 等於沒有報告。**

### $n$ 也依賴 $m_0$

還有一個更隱蔽的依賴。分支比計算的是「門檻以上的事件平均觸發幾個
門檻以上的事件」，兩個「門檻」都是 $m_0$。把門檻降低 $\delta$（新
門檻 $m_0'=m_0-\delta$）會怎樣？

兩件事同時發生。**一、每個親代被算到的後代變多了**：後代規模服從
GR 律，門檻降 $\delta$ 讓計數乘上 $e^{\beta\delta}$。**二、被拿來
平均的親代變小了**：新加入的親代都是小事件，產能較低，平均產能乘上
$e^{-\alpha\delta}$。淨效果是

$$n(m_0 - \delta) \;=\; n(m_0)\,e^{(\beta-\alpha)\delta} .$$

（完整推導見 13.11 節附錄 B。）由於 $\alpha<\beta$，這個因子大於 1
——**門檻降得愈低，分支比愈大**。取 $\beta-\alpha=0.6$、$\delta=1$，
因子是 1.82：$m_0$ 從 3.0 降到 2.0，$n$ 從 0.5 變成 0.91。

這條式子有一個前提必須講清楚：它假設**同一個自相似過程一路延伸到
門檻以下**，也就是門檻以下的小地震確實存在、也確實會觸發。固定 $m_0$
的模型對更小的事件其實什麼也沒說，所以這是一個外推，不是模型的推論。
但這個外推正是「地震的觸發到底從多小開始」這個問題的入口——如果一路
外推下去 $n$ 會超過 1，那就代表要嘛有一個最小觸發規模，要嘛 $\alpha$
其實更接近 $\beta$。

實務上的結論只有一句：**沒有宣告 $m_0$（以及 $\alpha=\beta$ 時的
$M_{\max}$）的分支比是一個沒有意義的數字**。文獻上各地估出的 $n$ 多在
0.3–0.9 之間，這個範圍有相當一部分來自門檻選擇，而不是構造差異。
唯一的例外是 $\alpha=\beta$：此時指數為零，$n$ 與門檻無關——這是自
相似設定的另一個好處，也是 simplETAS 的第二個賣點。

## 13.4 世代分解與臨界性

分支比是「每個事件平均生幾個」。把這個數字沿著世代往下累加，就得到
整個目錄的結構。

### 幾何級數

設觀測窗內的背景事件（第 0 代）期望數為 $N_0$。第 0 代每個事件平均生
$n$ 個直接後代，所以第 1 代的期望數是

$$N_1 = N_0 \cdot n .$$

這一步要用到一件事：**後代的規模與親代無關**（可分離性），所以第 1 代
事件的規模分布跟第 0 代一樣是 $s(m)$，它們的平均產能也一樣是 $n$。
於是同樣的關係可以一路遞推：

$$N_k = N_{k-1}\cdot n = N_0\,n^{k} .$$

總期望數是所有世代相加：

$$N_{\rm tot} = \sum_{k=0}^{\infty} N_0\,n^{k}
  = N_0 \sum_{k=0}^{\infty} n^{k} .$$

這個級數的和可以用一行代數算出來。取部分和 $S_K=\sum_{k=0}^{K}n^k$，
兩邊乘 $n$ 再相減：

$$\begin{aligned}
S_K &= 1 + n + n^2 + \cdots + n^{K}, \\
n\,S_K &= \phantom{1 + {}} n + n^2 + \cdots + n^{K} + n^{K+1}, \\
S_K - n\,S_K &= 1 - n^{K+1}
  \quad\Longrightarrow\quad
  S_K = \frac{1 - n^{K+1}}{1-n} .
\end{aligned}$$

中間所有項成對消掉，只剩頭尾。$n<1$ 時 $n^{K+1}\to0$，於是

$$\sum_{k=0}^{\infty} n^{k} = \frac{1}{1-n}
  \qquad\Longrightarrow\qquad
  N_{\rm tot} = \frac{N_0}{1-n} .$$

**這是整個 ETAS 最實用的一條式子。** 它說背景率被觸發機制放大了
$1/(1-n)$ 倍：$n=0.5$ 放大 2 倍，$n=0.9$ 放大 10 倍，$n=0.99$ 放大
100 倍。放大倍率對 $n$ 極度敏感，而且是在 $n$ 接近 1 時才變得敏感
——這是所有近臨界系統的共同特徵。

### 被觸發事件的比例恰好是 $n$

順手就得到第二條結果。目錄裡被觸發（非背景）的事件比例是

$$\frac{N_{\rm tot} - N_0}{N_{\rm tot}}
  = 1 - \frac{N_0}{N_{\rm tot}}
  = 1 - (1-n) = n .$$

乾淨到有點不可思議：**分支比同時是「平均生幾個」與「目錄裡有多大比例
是餘震」**。$n=0.5$ 代表一半的地震是被觸發的；$n=0.9$ 代表九成是。
這解釋了為什麼「一半以上的地震是餘震」在多數地區都成立——那不是一個
獨立的觀察，而是分支比在 0.5 以上的直接後果。

兩個必要的但書。**其一，這是穩態的期望值**，單一有限窗口的實現值會
有相當大的漲落，$n$ 愈接近 1 漲落愈大。**其二，「背景」是模型定義的，
不是物理定義的**：模型說某個事件沒被目錄裡的任何事件觸發，只代表它
沒被**記錄在目錄裡**的事件觸發；它可能被一個 $m<m_0$ 的事件、或一個
慢滑移事件推出來。這是 13.3 節門檻依賴性的另一面。

### 三種臨界狀態

| 狀態 | 條件 | 序列樣貌 | 模型是否可用 |
|---|---|---|---|
| 次臨界 | $n<1$ | 級聯必然熄滅，期望總數 $N_0/(1-n)$ | 可用 |
| 臨界 | $n=1$ | 家族大小無限期望值，序列可拖極長 | 邊界 |
| 超臨界 | $n>1$ | 級聯以正機率永不熄滅 | 概似積分發散 |

超臨界不只是「地球不長那樣」的問題，而是**模型本身壞掉**：
{eq}`eq:pp-loglik` 的第二項 $\int\lambda^*$ 在 $n>1$ 時發散，最大概似
估計沒有定義。所以擬合結果吐出 $\hat n>1$ 時，正確的反應不是宣布
「發現超臨界地殼」，而是回頭檢查 $m_0$、目錄完整度與空間範圍。

真實估計值多落在 0.3–0.9，看起來離 1 還有距離；但要記得 $1/(1-n)$ 的
放大倍率在這個區間已經從 1.4 走到 10。**「次臨界但接近臨界」正是為
什麼一次大地震之後的序列又長又猛、卻終究會停。** 用模擬直接看這個
差別：同樣的背景率，只把 $n$ 從 0.5 調到 0.9：

In [ ]:
fig = go.Figure()
MU3, ALPHA3, T3, N_REP = 0.4, 1.2, 365.0, 25
amp = []
for n_r, color in [(0.5, PALETTE[2]), (0.9, PALETTE[1])]:
    tot = bgs = 0
    for r in range(N_REP):                    # 平均倍率取多份實現
        c, _ = simulate_etas(MU3, n_r, ALPHA3, T3, seed=800 + r)
        tot += len(c)
        bgs += int((c.gen == 0).sum())
    amp.append(tot / bgs)
    cat_n, _ = simulate_etas(MU3, n_r, ALPHA3, T3, seed=800)   # 畫其中一份
    daily, _ = np.histogram(cat_n.t, bins=np.arange(0, int(T3) + 1))
    fig.add_trace(go.Scatter(
        x=np.arange(int(T3)), y=daily, mode="lines",
        name=f"n = {n_r}（此實現共 {len(cat_n)} 個，背景 "
             f"{int((cat_n.gen == 0).sum())} 個）",
        line=dict(color=color, width=1.1)))
apply_layout(fig, height=440, hovermode="x",
             xaxis_title="時間（天）", yaxis_title="每日事件數",
             yaxis_type="log",
             title=f"接近臨界的世界：{N_REP} 份模擬的平均放大倍率 "
                   f"{amp[0]:.2f}（n = 0.5，理論 2.0）對 "
                   f"{amp[1]:.2f}（n = 0.9，理論 10.0）")
fig

$n=0.5$ 的世界裡，級聯短促，活動大致貼著背景率起伏，偶爾冒出一個小
尖峰；$n=0.9$ 的世界裡，同樣數量的背景事件三不五時引爆連鎖，單日事件
數可以衝高一兩個數量級，而且高活動期一拖就是好幾天。

圖說裡的「平均放大倍率」是 25 份模擬目錄的總事件數除以總背景事件數，
要對照的理論值是 $1/(1-n)$。$n=0.5$ 幾乎吻合，$n=0.9$ 則明顯偏低。
原因有二。**一是有限觀測窗**：靠近 $T$ 的事件，它們的後代大量落在窗外
被丟掉，而 $n$ 愈大，被截掉的高階世代佔比就愈重。**二是單一實現的
漲落極大**：附錄 C 會算出家族大小的變異數是 $n/(1-n)^3$，在 $n\to1$
時以三次方發散，所以就算平均值對了，任何一份目錄看起來都可能離譜。

這兩件事都不是 bug，而是「有限目錄」的真實處境——第 14 章估參數時，
同樣的兩個效應會變成邊界效應與估計誤差。順帶一提，這也是為什麼
**不能拿一份目錄的「餘震佔比」直接當成 $\hat n$**：那個比值同時被
窗長、$m_0$ 與運氣決定。

### 把世代拆開看

上面的推導把世代當成記帳單位，但模擬時我們其實知道每個事件是第幾代。
把它們分層畫出來，$1/(1-n)$ 這條式子就從代數變成了圖：

In [ ]:
edges = np.arange(0, T1 + 1, 1.0)
gcap = np.minimum(cat1.gen.to_numpy(), 3)
fig = go.Figure()
frac = []
for g, name, color in [(0, "第 0 代（背景）", PALETTE[0]),
                       (1, "第 1 代", PALETTE[2]),
                       (2, "第 2 代", PALETTE[3]),
                       (3, "第 3 代以後", PALETTE[1])]:
    h, _ = np.histogram(cat1.t.to_numpy()[gcap == g], bins=edges)
    frac.append(int((gcap == g).sum()))
    fig.add_trace(go.Scatter(x=edges[:-1], y=h, mode="lines", name=name,
                             stackgroup="one", line=dict(width=0.6,
                                                         color=color),
                             fillcolor=color))
apply_layout(fig, height=440, hovermode="x",
             xaxis_title="時間（天）", yaxis_title="每日事件數（堆疊）",
             title=f"世代分解（n = {N1}）：背景 {frac[0]}、第 1 代 {frac[1]}、"
                   f"第 2 代 {frac[2]}、第 3 代以後 {frac[3]} 個")
fig

三件事可以直接讀出來。

**第一，背景那一層（最底下）是平的。** 它就是一條率為 $\mu$ 的齊次
Poisson 過程，沒有任何叢集結構。所有的尖峰、所有的「序列」，全部來自
上面那幾層。

**第二，尖峰是由高階世代堆出來的。** 每一次爆發，最厚的一層往往不是
第 1 代而是第 2 代以後。這是二次餘震的直接視覺證據：一個大事件的直接
後代裡若出現另一個大事件，它自己的後代就把尖峰再墊高一層。

**第三，各世代的事件數大致成幾何遞減**，比值就是 $n$。圖說裡的前三個
數字可以自己驗算一次：相鄰兩代的比值落在 0.8 附近，比 $n=0.85$ 略小，
一樣是有限窗口的邊界效應——高階世代出生得晚，被 $T$ 截掉的比例較高。
（第四個數字是「第 3 代以後」的**累計**，不能拿來算比值；它之所以比
第 2 代還大，正是因為它把後面所有世代都加了進來——這份目錄裡最深的
家族一路長到第十幾代。）

順帶說一件在真實資料上做不到的事：**世代標籤在真實目錄裡不存在**。
我們永遠無法知道某個地震「是第幾代」，最多只能像第 14 章那樣算出
「它是背景事件的機率 $\phi_j$」與「它被事件 $i$ 觸發的機率
$\rho_{ij}$」。這張圖是模擬才有的特權，也正是模擬在教學上不可取代的
理由。

## 13.5 前震是湧現的

現在來處理 ETAS 最反直覺、也最有威力的一個後果。

回到可分離性：後代的規模從 $s(m)$ 獨立抽出，**與親代規模無關**。模型
對後代規模的唯一約束是「發生在親代之後」，規模上沒有任何上限。所以
ETAS 天然會產生「後代比親代大」的序列。當這種事發生，我們事後回頭看
那個較小的親代，就叫它**前震**。

**ETAS 沒有前震機制，前震卻自動出現。** 這句話可以量化，而且量化的
過程很短。

### 直接後代：一個精確結果

設某個事件的規模是 $m_1$，記 $u_1 = m_1-m_0$。它的直接後代數服從
$\mathrm{Poisson}\bigl(\kappa(m_1)\bigr)$，每個後代的規模獨立抽自
$s(m)$，因此單一後代超過親代的機率是

$$\theta \;=\; P(M > m_1) \;=\; \int_{m_1}^{\infty}\beta
  e^{-\beta(m-m_0)}\,\mathrm{d}m \;=\; e^{-\beta u_1} .$$

接下來用 Poisson 過程的**稀疏化**性質：從 $\mathrm{Poisson}(\Lambda)$
個點中，各自以機率 $\theta$ 獨立保留，保留下來的個數服從
$\mathrm{Poisson}(\Lambda\theta)$。（這是 10.6 節 thinning 論證的離散
版本，證明只要把 Poisson 機率質量函數代進去重排即可。）於是

記 $N_{>}$ 為「比親代大的直接後代個數」，則

$$\begin{aligned}
N_{>} &\sim \mathrm{Poisson}\bigl(\kappa(m_1)\,\theta\bigr), \\
\kappa(m_1)\,\theta &= A\,e^{\alpha u_1}\cdot e^{-\beta u_1}
  = A\,e^{-(\beta-\alpha)u_1} .
\end{aligned}$$

取「至少一個」的機率（Poisson 取 0 的機率是 $e^{-\Lambda}$）：

$$P\bigl(N_{>} \ge 1\bigr)
  \;=\; 1 - \exp\!\left[-A\,e^{-(\beta-\alpha)(m_1-m_0)}\right] .$$

這條式子短，但每一個符號都在說話：

- **$\alpha<\beta$**：指數為負，機率隨 $m_1$ 遞減。大地震很難被自己的
  後代蓋過去——符合直覺，也是「主震–餘震」這個語彙為什麼堪用的原因。
- **$\alpha=\beta$**：指數歸零，$P = 1-e^{-A}$，**與親代規模完全無關**。
  任何規模的事件，成為「前震」的機率都一樣。這是完全自相似的直接表現。
- **$\alpha>\beta$**：機率隨 $m_1$ 遞增，大地震反而更容易被後代蓋過
  ——這已經不是我們認識的地球了，也再次呼應 13.3 節的收斂條件。

中間那一條正是{doc}`第 12 章 <12_clustering_laws>`的 Båth 定律接進來
的地方。Båth 說最大餘震與主震的規模差 $\Delta_1$ 大致是一個常數，
**與主震規模無關**。ETAS 要重現這種「與規模無關」，最自然的參數化就是
$\alpha=\beta$：此時整個觸發結構在規模軸上平移不變，$\Delta_1$ 的分布
自然不依賴主震規模。這是 simplETAS 把 $\alpha$ 釘在 $\beta$ 的第三個
理由（前兩個是自相似性與「修正不完整性後 $\alpha$ 本來就接近 $\beta$」）。

反過來，12.5 節的極值論證也在同一件事上：假設序列裡所有事件的規模是
同一個 GR 分布的獨立抽樣，最大與次大之差的期望值是 $1/\beta$。ETAS 的
可分離性假設正是「同一個 GR 分布的獨立抽樣」，所以第 12 章那一整套
順序統計量的結果可以直接搬進 ETAS，不必重推。

### 整個家族：從直接後代到所有後代

上面算的是**直接**後代。但「前震」的日常用法通常指整個序列：某個事件
之後不久發生了更大的事件，中間隔了幾代都算。要算這個，需要整個家族的
大小分布。

把問題簡化成平均場（mean-field）版本：假設每個事件的後代數都服從
$\mathrm{Poisson}(n)$，忽略「大事件後代多」這層相關。設 $Y$ 是一個
家族的總事件數（含始祖），它的機率生成函數 $H(z)=E[z^Y]$ 滿足

$$H(z) = z\,e^{\,n\,(H(z)-1)}$$

（推導見 13.11 節附錄 C：始祖貢獻一個 $z$，它的每個後代各自開啟一個
統計上完全相同的子家族，Poisson 的生成函數是 $e^{n(s-1)}$）。後代數
$Z = Y-1$ 的生成函數就是 $H(z)/z$。

「家族裡至少一個成員比始祖大」的機率，是對後代數取期望：給定 $Z=k$，
沒有任何一個超過始祖的機率是 $(1-\theta)^k$，所以

$$P \;=\; 1 - E\bigl[(1-\theta)^{Z}\bigr]
  \;=\; 1 - \frac{H(1-\theta)}{1-\theta} .$$

令 $w = H(1-\theta)$，它滿足 $w = (1-\theta)\,e^{n(w-1)}$，這是一個
一維方程式，數值上迭代幾次就收斂（$n<1$ 保證它是壓縮映射）。

**小 $\theta$ 的極限特別好記。** 令 $w = 1-\varepsilon$ 並展開到一階：

$$\begin{aligned}
1-\varepsilon &= (1-\theta)\,e^{-n\varepsilon}
  \;\approx\; (1-\theta)(1-n\varepsilon) \\
  &\approx\; 1 - n\varepsilon - \theta
\quad\Longrightarrow\quad
\varepsilon \approx \frac{\theta}{1-n}, \\
P &= 1 - \frac{1-\varepsilon}{1-\theta}
  \;\approx\; \varepsilon - \theta
  \;=\; \frac{n}{1-n}\,\theta .
\end{aligned}$$

結果漂亮得可以直接用直覺解釋：$n/(1-n)$ 恰好是**平均後代數**
（$E[Z] = 1/(1-n) - 1$），$\theta$ 是每個後代超過始祖的機率，兩者相乘
就是「期望有幾個成員比始祖大」。小機率時「至少一個」約等於期望個數。

代回 $\theta = e^{-\beta(m_1-m_0)}$：

$$P \;\approx\; \frac{n}{1-n}\,e^{-\beta(m_1-m_0)} .$$

**$n\to1$ 時這個機率發散（意思是不再是小量，近似失效）。** 也就是說，
愈接近臨界，「前震」愈是家常便飯。這給了一個實質的物理陳述：前震現象
的普遍程度，不需要任何前震機制來解釋，只需要一個接近 1 的分支比。

先把家族結構直接畫出來。模擬時我們記錄了每個事件的親代與所屬家族，
把親子連線畫上去：

In [ ]:
tv, mv, pv = cat1.t.to_numpy(), cat1.m.to_numpy(), cat1.parent.to_numpy()
big = int(np.argmax(mv))
founder = int(cat1.root[big])                 # 該家族的始祖（背景事件）
fam_mask = cat1.root.to_numpy() == founder

edge_x, edge_y = [], []
for k in np.flatnonzero(pv >= 0):
    edge_x += [tv[pv[k]], tv[k], None]
    edge_y += [mv[pv[k]], mv[k], None]

fig = go.Figure()
fig.add_trace(go.Scattergl(x=edge_x, y=edge_y, mode="lines", name="親 → 子",
                           line=dict(color="#bbbbbb", width=0.7)))
for sel, name, color in [(~fam_mask, "其他事件", ACCENT),
                         (fam_mask, "最大事件所屬家族", QUAKE_COLOR)]:
    fig.add_trace(go.Scattergl(
        x=tv[sel], y=mv[sel], mode="markers", name=name,
        marker=dict(size=4 + (mv[sel] - M0) * 3.5, color=color, opacity=0.75)))
fig.add_trace(go.Scatter(
    x=[tv[founder]], y=[mv[founder]], mode="markers", name="該家族的始祖",
    marker=dict(size=16, color="rgba(0,0,0,0)", symbol="circle",
                line=dict(color="#333333", width=2))))
apply_layout(fig, height=470, hovermode="closest",
             xaxis_title="時間（天）", yaxis_title="規模",
             title=f"觸發樹：最大事件 M {mv[big]:.2f}（第 {int(cat1.gen[big])} "
                   f"代）與它的家族（共 {int(fam_mask.sum())} 個事件，"
                   f"始祖 M {mv[founder]:.2f}）")
fig

灰線是親子關係，紅點是模擬中最大事件所屬的整個家族，黑圈標出的是這個
家族的**始祖**（第 0 代的背景事件）。兩件事值得看清楚：

**一、家族可以枝繁葉茂好幾代。** 這一個家族就佔了整份目錄兩成左右的
事件，而且深度遠不止兩三代。二次、三次餘震不是特例而是常態，這與 13.4
節的世代分解圖是同一件事的兩種畫法。

**二、黑圈不是最高的那個點。** 這份模擬裡最大的事件是第 2 代，它的
始祖只是一個規模剛過門檻的背景事件——在傳統語彙裡，那個較小的祖先
（以及中間那一代）就叫「前震」。注意這裡沒有任何前震機制，只有
「後代規模獨立於親代」這一條假設。

現在把第二點量化。掃一遍不同的分支比，統計「家族中最大的事件不是
始祖」的比例，並與上面推導的平均場理論值比較。注意這個統計量**只
依賴族譜與規模**，跟事件發生在什麼時候、什麼地方無關，所以下面的
模擬乾脆把時間與空間整個丟掉，只長家族樹——這樣就沒有觀測窗截斷的
問題，可以乾淨地對上理論（$\alpha$ 固定在 1.2，接近台灣的估計值）：

In [ ]:
def theory_not_founder(n_ratio, n_theta=400):
    """平均場理論：解 w = (1−θ)exp[n(w−1)]，再對 θ ~ U(0,1) 平均。

    截斷 GR 之下 θ = P(M > m1) 幾乎就是 U(0,1)，故直接在 θ 上積分。
    """
    th = (np.arange(n_theta) + 0.5) / n_theta
    w = np.zeros_like(th)
    for _ in range(200):                      # n < 1 保證是壓縮映射
        w = (1 - th) * np.exp(n_ratio * (w - 1))
    return float(np.mean(1 - w / (1 - th)))


def simulate_family(alpha, A, rng, cap=200_000):
    """只追蹤族譜與規模（時間、空間與觀測窗都不影響這個統計量）。

    回傳 (始祖規模, 家族總成員數, 家族最大規模)。
    """
    m_root = float(draw_mag(rng, 1)[0])
    queue, size, m_top = [m_root], 1, m_root
    while queue:
        mp = queue.pop()
        k = rng.poisson(A * np.exp(alpha * (mp - M0)))
        if k == 0:
            continue
        kids = draw_mag(rng, k)
        size += k
        m_top = max(m_top, float(kids.max()))
        queue.extend(kids.tolist())
        if size > cap:
            break
    return m_root, size, m_top


n_grid = np.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9])
N_FAM, ALPHA_F = 12000, 1.2
emp_all, emp_big, theo = [], [], []
for j, n_r in enumerate(n_grid):
    rng = np.random.default_rng(3000 + j)
    A_f = A_from_n(n_r, ALPHA_F)
    fam = np.array([simulate_family(ALPHA_F, A_f, rng) for _ in range(N_FAM)])
    hit = fam[:, 2] > fam[:, 0] + 1e-12
    big_fam = fam[:, 1] >= 10
    emp_all.append(float(hit.mean()))
    emp_big.append(float(hit[big_fam].mean())
                   if big_fam.sum() >= 60 else np.nan)
    theo.append(theory_not_founder(n_r))

fig = go.Figure()
fig.add_trace(go.Scatter(x=n_grid, y=theo, mode="lines",
                         name="平均場理論（全部家族）",
                         line=dict(color="#888888", width=2, dash="dash")))
fig.add_trace(go.Scatter(x=n_grid, y=emp_all, mode="lines+markers",
                         name="模擬：全部家族",
                         line=dict(color=ACCENT, width=2),
                         marker=dict(size=8)))
fig.add_trace(go.Scatter(x=n_grid, y=emp_big, mode="lines+markers",
                         name="模擬：成員 ≥ 10 的家族",
                         line=dict(color=PALETTE[1], width=2),
                         marker=dict(size=8, symbol="square")))
apply_layout(fig, height=450, hovermode="x",
             xaxis_title="分支比 n", yaxis_title="家族最大事件不是始祖的比例",
             yaxis_range=[0, 1],
             title=f"「最大的不是第一個」：n = 0.5 時 {emp_all[2]:.0%}，"
                   f"n = 0.9 時 {emp_all[-1]:.0%}（大家族分別為 "
                   f"{emp_big[2]:.0%}、{emp_big[-1]:.0%}）")
fig

三條線一起讀。

**虛線是平均場理論**，也就是解 $w=(1-\theta)e^{n(w-1)}$ 之後對
$\theta\sim U(0,1)$ 平均的結果。它單調上升、在 $n\to1$ 時趨近 1，
與小 $\theta$ 近似 $P\approx\frac{n}{1-n}\theta$ 的方向一致。

**藍線是模擬的全部家族**（含只有一個成員的家族，那些一定算「始祖最
大」）。它與理論同方向、同量級，但系統性偏低。原因不是模擬錯了，而是
平均場丟掉了一層相關：**在真正的 ETAS 裡，家族大小與始祖規模是正相關
的**（$\alpha>0$，大始祖生得多）。於是「家族大」與「始祖難被超越」總
是綁在一起出現，「家族小」與「始祖容易被超越」也綁在一起，兩種情況都
讓「最大的不是始祖」變難。平均場假設每個家族的大小都是同一個
$\mathrm{Poisson}(n)$ 分支的結果，看不到這層負向抵銷。

**橘線是成員數 $\ge10$ 的家族**，比例高得多——這正是上面那句話的直接
驗證：條件在「家族夠大」上，就把負相關的一半拿掉了。橘線也更貼近實務
上會被稱為「序列」的對象：沒有人會把一個孤立事件叫做一場序列。

這張圖有一個直接的防災意涵。在 $n\approx0.9$ 的世界裡，一場有規模的
序列中，**最大的事件不是第一個事件**是常態而非例外。這正是為什麼
「先來一個 $M_L\,5$，要不要發布警示」這麼難回答——ETAS 能給的答案是
一個機率，而不是一個是非題。第 22 章的作業化預報會回到這個場景。

## 13.6 模擬：branching 演算法

{doc}`第 10 章 <10_point_process>`的 10.6 節介紹了兩條模擬路線：
thinning 對應**數學**（條件強度的
定義），branching 對應**物理故事**（誰觸發了誰）。ETAS 的教學一律用
後者，理由很簡單：branching 法的每一行都對得上一句話，而且它免費附送
每個事件的親代標籤——13.4 與 13.5 兩節的圖全部靠這個標籤才畫得出來。

### 虛擬碼

```text
輸入：mu(x,y), A, alpha, c, p, D, gamma, q, beta, m0, Mmax
      時間窗 [0, T]，空間窗 S

步驟 0（移民 / 第 0 代）
  N0 ~ Poisson( ∫∫ mu(x,y) dx dy · T )
  對每個背景事件：
      時間  t  ~ Uniform(0, T)
      位置 (x,y) ~ 密度正比於 mu(x,y)
      規模  m  ~ s(m)   （截斷 GR，反函數法）
  把它們放進待處理佇列，世代 = 0

步驟 1（繁殖）
  從佇列取出一個事件 i：
      k ~ Poisson( kappa(m_i) )            # 生幾個
      對每個後代 j = 1..k：
          dt ~ g(·)     反函數法：dt = c[(1−U)^(−1/(p−1)) − 1]
          r  ~ f(·) 的徑向分布：
               r = sqrt( sigma·[(1−U)^(−1/(q−1)) − 1] )
               sigma = D·exp(gamma·(m_i − m0))
          phi ~ Uniform(0, 2π)             # 各向同性
          t_j = t_i + dt
          (x_j, y_j) = (x_i + r cos phi, y_i + r sin phi)
          m_j ~ s(m)                       # 與親代規模無關
          若 t_j < T：記錄（親代 = i，世代 = 世代_i + 1）並放進佇列

步驟 2（遞迴）
  重複步驟 1 直到佇列清空（n < 1 保證必然發生）

步驟 3（收尾）
  丟掉落在 S 之外的事件
  丟掉 burn-in 期（起始時間往前多跑一段，避免邊界效應）
```

### 逐行對應的物理故事

**「$N_0\sim\mathrm{Poisson}$」**：背景事件是板塊加載慢慢累積出來的，
彼此獨立、與歷史無關——這正是齊次 Poisson 過程的定義。空間上不均勻
是因為構造不均勻，但時間上恆定。

**「$k\sim\mathrm{Poisson}(\kappa(m_i))$」**：一個事件的後代數是隨機
的，期望值由產能律決定。用 Poisson 而不是固定值，是因為條件強度模型
在數學上等價於「每個親代獨立產生一個 Poisson 數量的後代」（Hawkes &
Oakes 1974 的表現定理，10.6 節提過）。

**「$\mathrm{d}t$ 用反函數法從 $g$ 抽」**：這一行就是 10.6 節推導的
$\Delta t = c[(1-U)^{-1/(p-1)}-1]$，那裡也順帶說明了為什麼 $p>1$
是必要的——$p\le1$ 時指數的符號翻轉，公式失去意義。

**「半徑用反函數法、角度均勻抽」**：空間核各向同性，所以角度是
$U(0,2\pi)$；半徑的累積分布函數要帶 Jacobian $2\pi r$，10.6 節推過，
結果是 $r=\sqrt{\sigma[(1-U)^{-1/(q-1)}-1]}$，形式與時間核完全平行
（$t/c \leftrightarrow r^2/\sigma$，$p\leftrightarrow q$）。

**「$m_j\sim s(m)$，與親代規模無關」**：整個模型最強的一條假設，也是
13.5 節前震湧現的唯一來源。改掉這一行，ETAS 就不是 ETAS 了。

**「若 $t_j<T$」**：落在窗外的後代直接丟棄。這是模擬版的邊界效應
——它讓模擬目錄的事件數系統性低於 $N_0/(1-n)$，13.4 節的圖已經看到。
反過來，若要模擬得更真實，起始時間應該往前多跑一段 burn-in，讓窗
開始前發生的事件也有機會把後代送進窗內。

**「$n<1$ 保證佇列必然清空」**：這是 13.3、13.4 兩節唯一的實作意義。
若不小心設了 $n\ge1$，這個 while 迴圈會跑到記憶體爆掉。

### 一個參數換一種地震活動型態

演算法寫好之後，最值得做的實驗是掃 $\alpha$。固定分支比（也就是固定
「平均生幾個」），只改「規模怎麼換算成產能」：

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    subplot_titles=("α = 0.5：群震型（大小事件觸發力接近）",
                                    "α = 2.0：主震–餘震型（大事件主導）"))
share = []
for row, (al, color) in enumerate([(0.5, PALETTE[2]), (2.0, PALETTE[1])], 1):
    cat_a, _ = simulate_etas(mu=1.0, n_ratio=0.7, alpha=al, T=200.0, seed=23)
    par = cat_a.parent.to_numpy()
    trig = par >= 0                                  # 被觸發的事件
    cut = np.quantile(cat_a.m.to_numpy(), 0.95)      # 最大的 5% 事件
    top_idx = set(np.flatnonzero(cat_a.m.to_numpy() >= cut).tolist())
    by_top = sum(1 for p in par[trig] if int(p) in top_idx)
    share.append(by_top / max(1, trig.sum()))
    fig.add_trace(go.Scattergl(
        x=cat_a.t, y=cat_a.m, mode="markers",
        name=f"α = {al}（{len(cat_a)} 個事件）",
        marker=dict(size=4 + (cat_a.m - M0) * 3.5, color=color, opacity=0.7)),
        row=row, col=1)
fig.update_yaxes(title_text="規模", range=[M0 - 0.2, 7.4])
fig.update_xaxes(title_text="時間（天）", row=2, col=1)
apply_layout(fig, height=560, hovermode="closest",
             title=f"同樣的分支比 n = 0.7，只改 α：最大的 5% 事件"
                   f"直接觸發了 {share[0]:.0%}（α = 0.5）對 "
                   f"{share[1]:.0%}（α = 2.0）的被觸發事件")
fig

兩張圖的**平均生產力完全一樣**（$n=0.7$，也就是七成的事件是被觸發
的），連背景事件都因為共用亂數種子而完全相同，但長相截然不同。

$\alpha=0.5$ 時，大事件與小事件的觸發力差不多，於是觸發量被大量的小
事件**分攤**掉——圖說的統計量說得很白：規模排前 5% 的事件只直接觸發
了不到一成的被觸發事件。序列的樣子是「一團一團規模相近的事件」，
沒有明顯的主從關係——這就是**群震（swarm）**。台灣的龜山島、日本的
伊豆半島常見這種型態，機制上多半與流體或慢滑移有關。

$\alpha=2.0$ 時，觸發力壓倒性**集中**在少數大事件身上：同樣是前 5%
的事件，直接觸發了將近一半的被觸發事件。序列的樣子是「一個大事件
之後跟著一大群小事件、然後安靜下來」——標準的**主震–餘震型**。

這解釋了為什麼 $\alpha$ 是 ETAS 少數能直接對應「活動型態分類」的參數，
也解釋了為什麼把日本的 $\alpha$ 搬到台灣、或把台灣東部的 $\alpha$ 搬
到西部，都是危險的。

還有一個實務上的警告要先講：**目錄的早期不完整會系統性壓低
$\hat\alpha$**。大震剛過的幾小時是產能最高的時候，卻也正是小地震被
大波形淹沒、記錄不到的時候。這一段高產出被目錄「咬掉」，模型就會
以為大地震沒那麼多產。Hainzl et al.（2013）指出，同時處理不完整性與
時變背景率之後，$\hat\alpha$ 會回升到接近 $\beta$——這是第 14 章估計
篇的核心議題之一，也是 simplETAS 敢設 $\alpha=\beta$ 的第一個理由。

## 13.7 傳染病類比：好用在哪裡，壞在哪裡

ETAS 的 E 就是 epidemic。這個類比不是事後的比喻，而是模型的來源
——Ogata 借的正是傳染病學裡「傳染型」分支過程的骨架。對照表非常整齊：

| 傳染病 | ETAS | 共同的數學物件 |
|---|---|---|
| 基本再生數 $R_0$ | 分支比 $n$ | 分支過程的平均後代數 |
| 潛伏期分布 | Omori 核 $g(t)$ | 後代出生時刻的密度 |
| 傳播距離分布 | 空間核 $f(x,y;m)$ | 後代出生位置的密度 |
| 境外移入 | 背景事件 $\mu$ | 移民過程（Poisson） |
| 本土感染 | 被觸發事件 | 分支項 |
| 群聚感染事件 | 高產能的大事件 | $\kappa(m)$ 的尾巴 |

這個橋很好走，而且走過去之後有實質收穫：

**一、$R_0=1$ 的直覺可以直接搬。** 疫情會不會擴散、序列會不會熄滅，
是同一個判準。13.4 節的 $1/(1-n)$ 在傳染病學裡就是「一個境外移入平均
引發多少本土病例」。

**二、除叢問題有現成的翻譯。** 「這個病例是境外移入還是本土感染」與
「這個地震是背景還是餘震」是同一個問題，而傳染病學早就放棄了二分法、
改用機率——這正是第 14 章隨機除叢的做法。

**三、方法論可以互相搬運。** COVID-19 期間有大量研究直接用 Hawkes
過程建模病例序列，用的就是{eq}`eq:pp-loglik`那一套；金融的訂單流、
神經元的 spike train 也是（10.7 節）。你在地震學裡卡住的問題，可能
別的領域已經解過。

### 類比壞掉的地方

但類比在三個地方會壞掉，而且壞得很徹底。

**第一，也是最根本的：地震的後代可以比親代大。** 傳染病裡沒有對應
物——沒有人會說「這一代的感染比上一代更嚴重」是模型的結構性後果。
在 ETAS 裡這卻是可分離性的必然結論（13.5 節），而且正是前震現象的
來源。這一條差異把「主震」這個概念整個推翻，也讓 ETAS 與傳統的主震／
餘震框架根本不相容。

**第二，沒有易感者耗竭。** 傳染病的 $R_0$ 會隨疫情推進而下降，因為
易感者被消耗掉了（群體免疫）。ETAS 沒有這個機制：$\kappa(m)$ 只看
規模，不看「這塊地殼已經被搖過幾次」。物理上這顯然不對——應力被
釋放之後不會立刻回填——但 ETAS 完全不建模這件事。序列會停下來，
純粹是因為 Omori 核衰減與 $n<1$，不是因為「用完了」。

**第三，沒有干預。** 疫苗、封城可以改變 $R_0$；地震沒有任何對應物。
這聽起來是廢話，但它有一個實質後果：傳染病模型的價值有一半在於評估
干預效果，而 ETAS 的價值全部在於描述與預報，一分也不在控制。

還有一個較細的分歧：傳染病的「傳染」有明確的個體接觸機制，地震的
「觸發」則只是**統計依賴**。10.9 節警告過，$\lambda^*$ 上升不等於
「A 用應力把 B 推出來」；兩個事件也可能同被第三個因素（慢滑移、流體
遷移）驅動，統計上一樣呈現叢集。ETAS 的親子關係是模型的記帳方式，
不是物理鏈條。

## 13.8 ETAS 家族：一張表看懂各種變體

「ETAS」在文獻裡不是一個模型，而是一整個家族。認得它們的差別，讀論文
時才不會把不同東西當成同一個。

| 版本 | 相對於前一版加了什麼 | 自由參數 | 出處 |
|---|---|---|---|
| 時間型 ETAS | 把 Omori 核疊加 | $\mu,A,c,\alpha,p$ | Ogata 1988 |
| 時空 ETAS | 空間核（含橢圓矩陣 $S_j$） | 上列 + $D,q$ | Ogata 1998 |
| $\alpha$–$\gamma$ 解耦 | 空間尺度自產能獨立出來 | 上列 + $\gamma$ | Ogata & Zhuang 2006 |
| ETES | 同族、不同命名與實作慣例 | 同上 | Console & Murru 2001 |
| R–J | 只保留主震的餘震率（見 14.7） | $a,b,c,p$ | Reasenberg & Jones 1989 |
| simplETAS | 釘死七個叢集參數 | $\mu,A$ 兩個 | Mancini & Marzocchi 2023 |
| 斷層幾何版 | 空間核改沿破裂面拉長／各向異性 | 依實作而定 | Ogata 1998 起 |
| UCERF3-ETAS | 後代可落在具名斷層上並長成大破裂 | 掛在斷層系統模型上 | Field et al. 2017 |

幾點導讀。

**時間型與時空型的差別不只是「多兩個參數」。** 時間型 ETAS 對整個
研究區給一個率，適合分析單一序列；時空型才能回答 CSEP 檢驗要的
「哪一格、多少個」。兩者的 $\ln L$ 不能互相比較（10.2 節第三條警告）。

**ETES（Epidemic Type Earthquake Sequence）與 ETAS 是同一副骨架**，
名稱不同主要是歷史因素與實作慣例（背景率怎麼算、參數怎麼固定）。
看到 ETES 不要以為是另一個模型家族。

**R–J 是 ETAS 的退化特例，不是競爭者。** 它只承認一個主震、只描述
那個主震的餘震率，等於把 {eq}`eq:etas-intensity` 的求和號縮成一項、
並把背景率丟掉。它的優點是參數極少、震後幾分鐘就能上線，這也是為什麼
美國 USGS 現行的 OAF（Operational Aftershock Forecasting）系統仍
以它為引擎之一。完整的推導與它跟 ETAS 的對應關係留給 14.7 節。

**simplETAS 走的是相反方向。** 它把 $\{\alpha,p,c,D,\gamma,q,\beta\}$
七個描述叢集的參數全部釘在普世經驗值，只估背景率與產能尺度兩個明顯
與區域相關的參數。在義大利，它從「日」到「四個世紀」四個時間尺度的
檢驗全部通過。釘死每個參數的理由（以及這是 bias–variance 取捨的地震學
版本這件事）留給第 14 章。

**斷層幾何版與 UCERF3-ETAS 是往物理靠的兩步。** 前者只是把各向同性的
$f$ 換掉，模型結構不變；後者則把 ETAS 掛在一整套斷層系統模型上，讓
後代可以「落在」某條具名斷層並成長成大破裂。代價是模型不再是十行
公式，而是一個需要整個團隊維護的軟體系統。

## 13.9 常見誤解與陷阱

**一、「分支比是一個可以跨研究比較的數字」。** 不行。13.3 節證明了
$n$ 依賴 $m_0$（因子 $e^{(\beta-\alpha)\delta}$），$\alpha=\beta$ 時
還依賴 $M_{\max}$（改一級差三成）。看到「某地的分支比是 0.8」，先問
門檻是多少、上限取多少、$\alpha$ 與 $\beta$ 各是多少。這四個數字缺
任何一個，那個 0.8 都無法解讀。

**二、「$\hat n>1$ 代表發現了超臨界地殼」。** 不是。$n>1$ 時
{eq}`eq:pp-loglik` 的 $\int\lambda^*$ 發散，最大概似估計沒有定義，
所以這個數字本身就是「估計程序出問題」的訊號。實際上最常見的原因是
$m_0$ 取得太低（把不完整的小事件也吃進去）、研究區切得太小（外部
觸發被算成內部級聯），或目錄有規模尺度不一致。

**三、「ETAS 的餘震一定比主震小」。** 錯得最徹底的一條。ETAS 對後代
規模的唯一約束是「從 $s(m)$ 獨立抽出」，時間上要在親代之後，規模上
完全沒有上限。13.5 節給了精確的機率公式，該節的圖顯示在
$n\approx0.9$ 而且**家族夠大**（成員 $\ge10$）時，「家族最大者不是
始祖」的比例超過八成，是常態而非例外。這正是 ETAS 能自然
重現前震的原因，也是它與傳統主震／餘震框架的根本分歧。

**四、「$A$ 與 $K$ 可以直接比較」。** 13.2 節算過：$A/K=c^{1-p}/(p-1)$
落在 13–28 之間，而且依賴 $p$ 與 $c$。這件事在同一篇論文的不同表格
之間也會出事——有些作者在正文用正規化寫法、在附錄引用他人數值時用
Ogata 寫法。

**五、「$\alpha$ 大代表危險」。** $\alpha$ 大只代表大地震主導，序列
是典型的主震–餘震型；$\alpha$ 小的群震型序列反而更難預測，因為隨時
可能冒出一個與前面規模相當、甚至更大的事件。從預報難度看，小 $\alpha$
才是麻煩。

**六、「模擬出來的目錄就是預報」。** 一份模擬目錄是模型的一個**實現**，
不是預報。預報是把數千到數十萬份實現統計起來得到的機率分布（或直接
算 $\int\lambda^*$ 得到期望數）。單看一份模擬目錄裡有沒有 $M\,7$，
跟預報完全無關。

**七、「$\lambda^*$ 跳很高代表下一個地震會很大」。** 10.9 節第六條
已經警告過，在可分離假設下規模完全由 $s(m)$ 決定。$\lambda^*$ 高只
代表「接下來會有很多地震」；出現大事件的絕對機率確實跟著上升，但那
是數量效應，不是規模分布變了。

**八、「$1/(1-n)$ 說背景率被放大，所以 ETAS 高估了地震數」。** 不是
放大也不是高估——$\mu$ 本來就只描述背景那一層，$N_0/(1-n)$ 才是模型
預期的總數。混淆的來源通常是把某篇論文的 $\hat\mu$ 當成「該區的地震
率」去比對目錄，數字當然對不上。

## 13.10 研究前沿與未解問題

### ETAS 是描述叢集的語言，不是物理定律

這是關於 ETAS 最重要的一句定位，值得把支撐它的證據攤開來看。

**證據一：它的組件全都是經驗律。** Omori（1894）、Utsu–Seki（1955）、
Gutenberg–Richter（1944）都是先從資料看出來、再寫成公式的。ETAS 是
把三條經驗律組裝成一個自洽的隨機過程，不是從應力轉移或摩擦定律推導
出來的。Ogata & Zhuang（2006）自己就寫模型形式「based on empirical
laws」。

**證據二：函數形式靠 AIC 挑，不靠物理挑。** Ogata（1998）比較了三種
空間核（高斯型與兩種反冪次型），選中反冪次型的理由是 AIC 最小，不是
哪一個比較「對」。2006 年多加一個 $\gamma$ 也是同一套邏輯。函數形式
是可協商的。

**證據三：不同構造區的參數不同，卻沒有第一原理告訴你該是多少。**
群震 $\alpha\in[0.35,0.85]$ 對非群震 $\alpha\in[1.2,3.1]$；$D$ 從
加州的不到 0.1 km$^2$ 到隱沒帶的超過 20 km$^2$。而 simplETAS 又發現
把它們釘在中間值也堪用——「足夠有彈性去描述，但沒有唯一正確答案」，
這正是描述性語言的特徵。

**證據四：它對「主震」這種人為概念完全無感。** ETAS 只承認背景與被
觸發，不承認主震與餘震。把外部定義硬套上去，就會製造出第 12 章那種
30% 的 b 值偏差假訊號。

要補一句平衡：這不代表 ETAS 沒有物理內涵。$q=1.5$ 對應靜態應力隨
距離三次方衰減、遠場成分對應動態觸發，都是有物理根據的解讀。但這些
是**事後的物理詮釋**，不是模型的來源。

### 標準化基準版本的缺席

一個很少被明講、但實際影響巨大的問題：**沒有一個「標準 ETAS」**。

每一個研究團隊都有自己的變體：空間核各向同性還是橢圓、背景率用固定
網格還是變頻寬核、$m_0$ 怎麼選、邊界事件怎麼處理、$M_{\max}$ 取多少、
最佳化用哪個演算法與什麼初始值。這些選擇每一個都合理，合起來卻讓
「ETAS 的預報表現」變成一個無法複製的量：兩篇都說「用 ETAS」的論文，
跑出來的結果可以差很多，而讀者無從判斷差異來自模型還是來自實作。

這正是 simplETAS 要解決的問題——它的第一個用途就是當 **benchmark**：
把所有選擇釘死、程式碼公開，任何新模型都先跟它比。有了共同基準，
跨區域的 CSEP 實驗才談得上比較「真正的技巧增益」。這個方向目前才剛
開始，而台灣還沒有自己的公開基準版本，是一個明顯可做的題目。

### 其他開放問題

**背景率真的是常數嗎？** {eq}`eq:etas-intensity` 假設 $\mu(x,y)$ 不隨
時間變。但慢滑移事件、孔隙壓力擴散、水庫與注水都會讓「自發」地震率
在幾天到幾年的尺度上變動。時變背景率的 ETAS 已經有不少嘗試，難處在
於它與觸發項高度混淆：背景率上升與分支比上升在目錄上看起來很像。

**有沒有最小觸發規模？** 13.3 節的門檻依賴性指出，$n$ 隨門檻降低而
上升。若一路外推到 $m\to-\infty$ 而 $n$ 沒有上限，模型就會超臨界。
所以要嘛存在一個最小觸發規模，要嘛 $\alpha$ 實際上更接近 $\beta$。
這個問題目前沒有共識，而它的答案會直接改變我們對「地殼有多接近臨界」
的判斷。

**物理與統計的接縫。** 一條路是把 Coulomb 應力變化或 rate-and-state
摩擦律接進 ETAS 的觸發項；另一條路（10.7 節提過）是把 $\lambda^*$ 的
函數形式整個丟掉，改用神經網路把歷史編碼成向量再輸出強度。兩條路都
面臨同一個考驗：**要先贏過參數少得可憐的 simplETAS**。目前為止，
增加複雜度換來的技巧增益普遍不如預期。

## 13.11 附錄：本章推導細節

### A. 截斷分支比的完整代數與極限

截斷 GR 密度 $s_T(m)=\beta e^{-\beta u}/(1-e^{-\beta\Delta M})$，
$u=m-m_0\in[0,\Delta M]$。分支比

$$\begin{aligned}
n(\alpha,\Delta M)
  &= \frac{A\beta}{1-e^{-\beta\Delta M}}
     \int_0^{\Delta M} e^{-(\beta-\alpha)u}\,\mathrm{d}u \\
  &= \frac{A\beta}{1-e^{-\beta\Delta M}}\cdot
     \frac{1-e^{-(\beta-\alpha)\Delta M}}{\beta-\alpha} .
\end{aligned}$$

**極限一（$\alpha\to\beta$）**：令 $\epsilon=\beta-\alpha\to0$，用
$1-e^{-\epsilon\Delta M} = \epsilon\Delta M - \frac{(\epsilon\Delta
M)^2}{2}+O(\epsilon^3)$，

$$\frac{1-e^{-\epsilon\Delta M}}{\epsilon}
  \;\longrightarrow\; \Delta M
  \qquad\Longrightarrow\qquad
  n = \frac{A\beta\,\Delta M}{1-e^{-\beta\Delta M}} .$$

這是一個可去奇點，不是真的發散——未截斷版本之所以在 $\alpha=\beta$
爆掉，是因為 $\Delta M\to\infty$ 與 $\epsilon\to0$ 兩個極限不可交換。

**極限二（$\Delta M\to\infty$，$\alpha<\beta$）**：兩個指數項都趨於
零，回到 {eq}`eq:branching-ratio`。

**$\alpha>\beta$ 的行為**：此時 $\beta-\alpha<0$，
$1-e^{-(\beta-\alpha)\Delta M} = 1-e^{(\alpha-\beta)\Delta M}$ 為負，
除以負的 $(\beta-\alpha)$ 得正值，且隨 $\Delta M$ 呈**指數**成長
（$\propto e^{(\alpha-\beta)\Delta M}$）。也就是說 $\alpha>\beta$ 的
模型在數學上可用，但 $n$ 對 $M_{\max}$ 極度敏感，實務上不可靠。

### B. 分支比的門檻依賴性

設原門檻 $m_0$、參數 $(A,\alpha,\beta)$，新門檻 $m_0'=m_0-\delta$
（$\delta>0$）。假設同一個自相似結構延伸到 $m_0'$：規模密度仍是
$\beta e^{-\beta(m-m_0')}$，而規模 $m$ 的事件觸發「$\ge m_0'$ 的
後代」的期望數為

$$\kappa'(m) = \kappa(m)\,e^{\beta\delta}
  = A\,e^{\alpha(m-m_0)}e^{\beta\delta}$$

（降低門檻 $\delta$ 讓 GR 計數乘上 $e^{\beta\delta}$）。於是

$$\begin{aligned}
n' &= \int_{m_0'}^{\infty} A\,e^{\alpha(m-m_0)}e^{\beta\delta}\,
      \beta e^{-\beta(m-m_0')}\,\mathrm{d}m \\
   &= A\beta\,e^{\beta\delta}\int_{0}^{\infty}
      e^{\alpha(v-\delta)}e^{-\beta v}\,\mathrm{d}v
      \qquad (v = m - m_0') \\
   &= A\beta\,e^{(\beta-\alpha)\delta}\int_{0}^{\infty}
      e^{-(\beta-\alpha)v}\,\mathrm{d}v
    \;=\; \frac{A\beta}{\beta-\alpha}\,e^{(\beta-\alpha)\delta}
    \;=\; n\,e^{(\beta-\alpha)\delta} .
\end{aligned}$$

等價地，用門檻 $m_0'$ 表示的產能尺度是
$A' = \kappa'(m_0') = A\,e^{(\beta-\alpha)\delta}$，代進
{eq}`eq:branching-ratio` 得同一個結果。$\alpha=\beta$ 時指數為零，
$n$ 與門檻無關。

### C. 家族大小的生成函數（Borel 分布）

平均場設定：每個事件獨立產生 $\mathrm{Poisson}(n)$ 個後代。設 $Y$ 為
一個家族的總事件數（含始祖），$H(z)=E[z^Y]$。

對始祖的後代數 $N\sim\mathrm{Poisson}(n)$ 取條件：始祖本身貢獻一個
$z$，它的 $N$ 個後代各自開啟一個統計上與整個家族同分布的子家族，
彼此獨立。因此

$$\begin{aligned}
H(z) &= z\,E\bigl[H(z)^{N}\bigr]
     = z\sum_{k=0}^{\infty}\frac{e^{-n}n^{k}}{k!}H(z)^{k} \\
     &= z\,e^{-n}\,e^{\,n H(z)}
     = z\,e^{\,n(H(z)-1)} .
\end{aligned}$$

由 Lagrange 反演可得明確的機率質量函數（Borel 分布）

$$P(Y = k) = \frac{e^{-nk}(nk)^{k-1}}{k!},\qquad k = 1,2,\dots$$

其期望值 $E[Y]=1/(1-n)$——與 13.4 節用世代分解得到的結果一致，兩條
路徑互相驗證。變異數 $\mathrm{Var}[Y]=n/(1-n)^3$，在 $n\to1$ 時以
三次方發散：**這就是為什麼近臨界的目錄，單一實現的漲落大到讓人難以
相信它們來自同一個模型**。

### D. 家族中出現比始祖更大事件的機率

承附錄 C。後代數 $Z=Y-1$，其生成函數 $G_Z(z)=E[z^{Z}]=H(z)/z$。
每個後代獨立地以機率 $\theta$ 超過始祖，故

記 $P_0$ 為「家族中無人超過始祖」的機率、$P=1-P_0$ 為「至少一人超過」
的機率，則

$$\begin{aligned}
P_0 &= E\bigl[(1-\theta)^{Z}\bigr]
  = G_Z(1-\theta) = \frac{H(1-\theta)}{1-\theta}, \\
P &= 1 - \frac{w}{1-\theta},
  \qquad w \equiv H(1-\theta) .
\end{aligned}$$

$w$ 滿足 $w=(1-\theta)e^{n(w-1)}$。這個方程式在 $[0,1]$ 上有唯一解：
令 $F(w)=(1-\theta)e^{n(w-1)}$，則 $F$ 遞增、$F(0)>0$、
$F(1)=1-\theta<1$，且 $F'(w)=nF(w)\le n(1-\theta)<1$，是壓縮映射，
從任意起點迭代都收斂。本章圖 6 的理論曲線就是這樣算的。

**小 $\theta$ 展開**已在 13.5 節給出，結果是 $P\approx\theta\,n/(1-n)$。
**大 $\theta$ 的另一端**（$\theta\to1$，也就是始祖恰好在門檻上）：
$w\to0$ 而 $w/(1-\theta)\to e^{-n}$，故 $P\to1-e^{-n}$。這與 13.5 節
的**直接後代**公式在 $\alpha=\beta$、$A=n$ 時給的 $1-e^{-A}$ 一致
——因為門檻事件被超過的機率是 1，能不能被超過只取決於它有沒有後代。

### E. Poisson 稀疏化

13.5 節用到的性質：設 $N\sim\mathrm{Poisson}(\Lambda)$，對每個點獨立
以機率 $\theta$ 保留，保留數為 $N_1$。則

$$\begin{aligned}
P(N_1 = j) &= \sum_{k\ge j}\frac{e^{-\Lambda}\Lambda^{k}}{k!}
  \binom{k}{j}\theta^{j}(1-\theta)^{k-j} \\
  &= \frac{e^{-\Lambda}(\Lambda\theta)^{j}}{j!}
     \sum_{l\ge0}\frac{[\Lambda(1-\theta)]^{l}}{l!}
     \qquad (l = k-j) \\
  &= \frac{e^{-\Lambda\theta}(\Lambda\theta)^{j}}{j!} ,
\end{aligned}$$

即 $N_1\sim\mathrm{Poisson}(\Lambda\theta)$。第二個等號把二項式係數
拆開、把 $\Lambda^k$ 分成 $\Lambda^j\Lambda^l$；第三個等號用
$e^{\Lambda(1-\theta)}$ 吸收求和，與前面的 $e^{-\Lambda}$ 合併成
$e^{-\Lambda\theta}$。這個結果與 10.6 節 thinning 的幾何論證是同一
件事的離散版本。

---

回頭看，這一章其實只做了一件事：把{doc}`第 12 章 <12_clustering_laws>`
的三個零件放進{doc}`第 10 章 <10_point_process>`的求和號裡。但這一步
產生的東西遠超過投入：二次餘震、前震、幾何級數的放大倍率、一個決定
模型生死的積分收斂條件，全部是**湧現的**，沒有一項是額外寫進去的。
一個好的模型就該是這樣——你放進去三條經驗律，它還你一整套現象。

但模型寫得再漂亮，也還不能用。{eq}`eq:etas-intensity` 裡的八個參數
現在還是八個符號，而真實目錄有不完整、有邊界、有規模尺度不一致、
有高度相關的參數與貼在邊界上的最佳化。更麻煩的是，這一章一路假設的
「背景 vs 被觸發」在真實資料上沒有標籤——13.4 節那張世代分解圖是
模擬才有的特權。

{doc}`第 14 章 <14_etas_estimation>`要處理的正是這些事：把
{eq}`eq:pp-loglik` 實際算出來、用隨機除叢把「這是不是背景事件」變成
一個機率 $\phi_j$、看清楚 ETAS 參數之間的相關性有多恐怖，並回答一個
這一章刻意沒碰的問題——如果七個參數都可以釘死，我們到底在估什麼？